In [ ]:
# ============================================================
# CELL 1 - Imports and Parameters
# ============================================================
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from IPython.display import display

from ramp_prediction_helpers import (
    CLASS_CODE,
    CLASS_COLOR,
    FEATURE_COLS,
    GAP_COLS,
    N_BINS,
    PAD_COLS,
    PROFILE_COLS,
    build_prediction_dataset,
    apply_predictions_to_sidewalk,
    write_sidewalk_prediction_asc,
    write_sidewalk_prediction_csv,
    write_sidewalk_prediction_las,
)

# Stage 1 outputs for the zone/segment you want to predict.
SIDEWALK_CSV = Path(r"C:\Users\jianjing\Desktop\Fiseha\Ramp_Detection\11\segments\part1\traj_sidewalk.csv")
LANE_CSV = Path(r"C:\Users\jianjing\Desktop\Fiseha\Ramp_Detection\11\segments\part1\traj_lane.csv")

# Trained model outputs from pytorch_ramp_cnn_training.ipynb.
MODEL_DIR = Path(r"C:\Users\jianjing\Desktop\Fiseha\Ramp_Detection\training\old_dataset_v2\pytorch_ramp_cnn_outputs")
MODEL_PATH = MODEL_DIR / "ramp_cnn_pytorch.pt"
META_PATH = MODEL_DIR / "ramp_cnn_pytorch_meta.json"

# Output names. Only sidewalk point products are written in this prediction notebook.
# All prediction outputs are isolated in a predictions/ folder beside the input CSVs.
PREDICTIONS_DIR = SIDEWALK_CSV.parent / "predictions"
PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)
BASE_NAME = SIDEWALK_CSV.stem.replace("_sidewalk", "")
STATION_PREDICTIONS_CSV = PREDICTIONS_DIR / f"{BASE_NAME}_cnn_predictions.csv"
SIDEWALK_PRED_CSV = PREDICTIONS_DIR / f"{BASE_NAME}_sidewalk_cnn_pred.csv"
SIDEWALK_PRED_ASC = PREDICTIONS_DIR / f"{BASE_NAME}_sidewalk_cnn_pred_CC.asc"
SIDEWALK_PRED_LAS = PREDICTIONS_DIR / f"{BASE_NAME}_sidewalk_cnn_pred_CC.las"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Sidewalk CSV: {SIDEWALK_CSV}")
print(f"Lane CSV    : {LANE_CSV}")
print(f"Model path  : {MODEL_PATH}")
print(f"Meta path   : {META_PATH}")
print(f"Device      : {DEVICE}")
print()
print(f"Predictions dir: {PREDICTIONS_DIR}")
print(f"Station predictions CSV: {STATION_PREDICTIONS_CSV}")
print(f"Sidewalk prediction CSV: {SIDEWALK_PRED_CSV}")
print(f"Sidewalk prediction ASC: {SIDEWALK_PRED_ASC}")
print(f"Sidewalk prediction LAS: {SIDEWALK_PRED_LAS}")
print()
print(f"Expected model bins: {N_BINS}")
print(f"Feature columns: {FEATURE_COLS}")
print(f"Prediction color map: {CLASS_COLOR}")


Sidewalk CSV: C:\Users\jianjing\Desktop\Fiseha\Ramp_Detection\11\segments\part1\Sidewalk.las
Lane CSV    : C:\Users\jianjing\Desktop\Fiseha\Ramp_Detection\11\segments\part1\Lane.las
Model path  : C:\Users\jianjing\Desktop\Fiseha\Ramp_Detection\training\old_dataset_v2\pytorch_ramp_cnn_outputs\ramp_cnn_pytorch.pt
Meta path   : C:\Users\jianjing\Desktop\Fiseha\Ramp_Detection\training\old_dataset_v2\pytorch_ramp_cnn_outputs\ramp_cnn_pytorch_meta.json
Device      : cuda

Predictions dir: C:\Users\jianjing\Desktop\Fiseha\Ramp_Detection\11\segments\part1\predictions
Station predictions CSV: C:\Users\jianjing\Desktop\Fiseha\Ramp_Detection\11\segments\part1\predictions\Sidewalk_cnn_predictions.csv
Sidewalk prediction CSV: C:\Users\jianjing\Desktop\Fiseha\Ramp_Detection\11\segments\part1\predictions\Sidewalk_sidewalk_cnn_pred.csv
Sidewalk prediction ASC: C:\Users\jianjing\Desktop\Fiseha\Ramp_Detection\11\segments\part1\predictions\Sidewalk_sidewalk_cnn_pred_CC.asc
Sidewalk prediction LAS: C:\Use

In [2]:
# ============================================================
# CELL 2 - Load and Validate Stage 1 CSV Inputs
# ============================================================
REQ_SIDEWALK_COLS = ["x", "y", "z", "s_m", "v_m", "side", "class", "label"]
REQ_LANE_COLS = ["z", "s_m", "v_m", "side"]

if not SIDEWALK_CSV.exists():
    raise FileNotFoundError(f"Sidewalk CSV does not exist: {SIDEWALK_CSV}")
if not LANE_CSV.exists():
    raise FileNotFoundError(f"Lane CSV does not exist: {LANE_CSV}")
if not MODEL_PATH.exists():
    raise FileNotFoundError(f"Model file does not exist: {MODEL_PATH}")
if not META_PATH.exists():
    raise FileNotFoundError(f"Model metadata file does not exist: {META_PATH}")

df_sw = pd.read_csv(SIDEWALK_CSV)
df_lane = pd.read_csv(LANE_CSV)

missing_sw = [c for c in REQ_SIDEWALK_COLS if c not in df_sw.columns]
missing_lane = [c for c in REQ_LANE_COLS if c not in df_lane.columns]
if missing_sw:
    raise ValueError(f"Missing required sidewalk columns: {missing_sw}")
if missing_lane:
    raise ValueError(f"Missing required lane columns: {missing_lane}")

df_sw = df_sw.copy()
df_lane = df_lane.copy()
df_sw["side"] = df_sw["side"].astype(str).str.strip().str.upper()
df_lane["side"] = df_lane["side"].astype(str).str.strip().str.upper()
df_sw["_s_key"] = pd.to_numeric(df_sw["s_m"], errors="coerce").round(3)
df_lane["_s_key"] = pd.to_numeric(df_lane["s_m"], errors="coerce").round(3)

for col in ["x", "y", "z", "s_m", "v_m"]:
    df_sw[col] = pd.to_numeric(df_sw[col], errors="coerce")
for col in ["z", "s_m", "v_m"]:
    df_lane[col] = pd.to_numeric(df_lane[col], errors="coerce")
df_sw["class"] = pd.to_numeric(df_sw["class"], errors="coerce")
df_sw["label"] = df_sw["label"].astype(str).str.strip()

valid_sides = {"RIGHT", "LEFT"}
bad_sw_sides = sorted(set(df_sw["side"].dropna()) - valid_sides)
bad_lane_sides = sorted(set(df_lane["side"].dropna()) - valid_sides)
if bad_sw_sides:
    raise ValueError(f"Unexpected sidewalk side values: {bad_sw_sides}")
if bad_lane_sides:
    raise ValueError(f"Unexpected lane side values: {bad_lane_sides}")

sw_station_sides = df_sw[["_s_key", "side"]].drop_duplicates().shape[0]
lane_station_sides = df_lane[["_s_key", "side"]].drop_duplicates().shape[0]
missing_lane_keys = df_sw[["_s_key", "side"]].drop_duplicates().merge(
    df_lane[["_s_key", "side"]].drop_duplicates(),
    on=["_s_key", "side"],
    how="left",
    indicator=True,
)
missing_lane_keys = missing_lane_keys[missing_lane_keys["_merge"] == "left_only"]

print(f"Sidewalk rows loaded       : {len(df_sw):,}")
print(f"Lane rows loaded           : {len(df_lane):,}")
print(f"Sidewalk station-sides     : {sw_station_sides:,}")
print(f"Lane station-sides         : {lane_station_sides:,}")
print(f"Sidewalk station-sides without matching lane: {len(missing_lane_keys):,}")
print()
print("Original rule-based sidewalk class distribution:")
print(df_sw["label"].value_counts(dropna=False).to_string())
print()
print("Sidewalk side counts:")
print(df_sw["side"].value_counts(dropna=False).to_string())
print()
print("Input preview:")
df_sw[["x", "y", "z", "s_m", "v_m", "side", "class", "label"]].head()


UnicodeDecodeError: 'utf-8' codec can't decode byte 0xea in position 92: invalid continuation byte

In [ ]:
# ============================================================
# CELL 3 - Build Prediction-Time Model Input Rows
# ============================================================
# This mirrors cnn_dataset_from_remapped_and_lane_csv.ipynb, except labels
# are kept only as reference metadata and are not used by the CNN.
df_pred_input, build_stats = build_prediction_dataset(
    df_sw=df_sw,
    df_lane=df_lane,
    source_file=str(SIDEWALK_CSV.resolve()),
)

required_model_cols = PROFILE_COLS + GAP_COLS + PAD_COLS + FEATURE_COLS
missing_model_cols = [c for c in required_model_cols if c not in df_pred_input.columns]
if missing_model_cols:
    raise ValueError(f"Prediction input is missing model columns: {missing_model_cols[:10]}")

nan_feature_counts = df_pred_input[FEATURE_COLS].isna().sum()
if (nan_feature_counts > 0).any():
    print("WARNING: Missing scalar feature values detected. These must be handled before inference.")
    print(nan_feature_counts.to_string())

print("Group processing summary:")
for k, v in build_stats.items():
    print(f"  {k}: {v:,}")
print()
print(f"Prediction input shape: {df_pred_input.shape}")
print(f"Profile columns: {len(PROFILE_COLS)} | Gap columns: {len(GAP_COLS)} | Pad columns: {len(PAD_COLS)}")
print(f"Feature columns: {FEATURE_COLS}")
print()
print("Valid-bin summary:")
print(df_pred_input[["n_valid_bins", "last_valid_bin", "n_sidewalk_pts", "n_lane_pts"]].describe().round(2).to_string())
print()
print("Original label distribution in prediction input:")
print(df_pred_input["original_label"].value_counts(dropna=False).to_string())
print()
print("Rule-function label distribution used only for feature auditing:")
print(df_pred_input["feature_rule_label"].value_counts(dropna=False).to_string())
print()
print("Scalar feature summary:")
print(df_pred_input[FEATURE_COLS].describe().round(5).to_string())
print()
buffer_cols = ["sidewalk_edge_buffer_m", "buffer_applied", "buffer_start_v", "buffer_end_v", "buffer_width_m", "sidewalk_edge_center_v", "sidewalk_edge_center_bin"]
available_buffer_cols = [c for c in buffer_cols if c in df_pred_input.columns]
if available_buffer_cols:
    print("Profile mode counts:")
    print(df_pred_input["profile_mode"].value_counts(dropna=False).to_string())
    print()
    print("Sidewalk-edge buffer summary:")
    print(df_pred_input[available_buffer_cols].describe().round(3).to_string())
    print()
edge_cols = ["lane_edge_v", "side_edge_v", "interface_center_v"]
available_edge_cols = [c for c in edge_cols if c in df_pred_input.columns]
if available_edge_cols:
    print("Edge metadata summary:")
    print(df_pred_input[available_edge_cols].describe().round(3).to_string())

preview_cols = ["s_m", "side", "original_label", "original_class", "feature_rule_label", "feat_kink_dz", "feat_kink_slope", "n_valid_bins", "last_valid_bin"]
preview_cols += [c for c in ["profile_mode"] + buffer_cols if c in df_pred_input.columns]
preview_cols += [c for c in edge_cols if c in df_pred_input.columns]
df_pred_input[preview_cols].head()


In [ ]:
# ============================================================
# CELL 4 - Split CNN-Eligible Rows From Rule-Based Passthrough Rows
# ============================================================
# The CNN was trained only on RAMP / CURB_NO_RAMP / DEPRESSED_DITCH.
# Rows without kink features, typically NO_SIDEWALK or SEPARATED_NO_RAMP,
# bypass the CNN and keep their original Stage 1 labels in the final output.
cnn_ready_mask = df_pred_input[FEATURE_COLS].notna().all(axis=1)

df_model_input = df_pred_input.loc[cnn_ready_mask].copy().reset_index(drop=True)
df_passthrough = df_pred_input.loc[~cnn_ready_mask].copy().reset_index(drop=True)

# Pre-fill passthrough prediction columns so later cells can concatenate
# passthrough rows with model-predicted rows without special casing.
if not df_passthrough.empty:
    df_passthrough["pred_label"] = df_passthrough["original_label"].fillna(df_passthrough["feature_rule_label"])
    df_passthrough["pred_class"] = pd.to_numeric(
        df_passthrough["original_class"], errors="coerce"
    ).fillna(df_passthrough["pred_label"].map(CLASS_CODE)).astype(int)
    df_passthrough["pred_confidence"] = 1.0
    df_passthrough["prob_RAMP"] = 0.0
    df_passthrough["prob_CURB_NO_RAMP"] = 0.0
    df_passthrough["prob_DEPRESSED_DITCH"] = 0.0
    df_passthrough["prediction_source"] = "passthrough_rule_based"
else:
    for col in [
        "pred_label",
        "pred_class",
        "pred_confidence",
        "prob_RAMP",
        "prob_CURB_NO_RAMP",
        "prob_DEPRESSED_DITCH",
        "prediction_source",
    ]:
        df_passthrough[col] = pd.Series(dtype=object)

print(f"Total prediction input rows : {len(df_pred_input):,}")
print(f"CNN-eligible rows          : {len(df_model_input):,}")
print(f"Passthrough rows           : {len(df_passthrough):,}")
print()
print("CNN-eligible original labels:")
print(df_model_input["original_label"].value_counts(dropna=False).to_string())
print()
print("Passthrough labels:")
if df_passthrough.empty:
    print("  none")
else:
    print(df_passthrough["pred_label"].value_counts(dropna=False).to_string())
    display_cols = [
        "s_m",
        "side",
        "original_label",
        "original_class",
        "pred_label",
        "pred_class",
        "prediction_source",
        "n_sidewalk_pts",
        "n_lane_pts",
    ]
    display(df_passthrough[display_cols])


In [ ]:
# ============================================================
# CELL 5 - Define Model Architecture and Load Trained Weights
# ============================================================
class SpatialAttention1d(nn.Module):
    def __init__(self, in_ch, reduction=8):
        super().__init__()
        self.attn = nn.Sequential(
            nn.Conv1d(in_ch, in_ch // reduction, kernel_size=1),
            nn.LeakyReLU(0.1),
            nn.Conv1d(in_ch // reduction, 1, kernel_size=1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        w = self.attn(x)
        return x * w


class RampCNN(nn.Module):
    def __init__(self, in_channels=3, num_features=2, num_classes=3, dropout=0.1):
        super().__init__()
        self.use_scalar_features = num_features > 0
        self.attn = SpatialAttention1d(128, reduction=8)
        self.features = nn.Sequential(
            nn.Conv1d(in_channels, 64, kernel_size=7, padding="same"),
            nn.BatchNorm1d(64),
            nn.LeakyReLU(0.1),
            nn.MaxPool1d(2),
            nn.Conv1d(64, 128, kernel_size=5, padding="same"),
            nn.BatchNorm1d(128),
            nn.LeakyReLU(0.1),
            nn.MaxPool1d(2),
            nn.Conv1d(128, 128, kernel_size=3, padding="same"),
            nn.BatchNorm1d(128),
            nn.LeakyReLU(0.1),
        )
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.profile_head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 64),
            nn.LeakyReLU(0.1),
        )
        if self.use_scalar_features:
            self.feature_head = nn.Sequential(
                nn.Linear(num_features, 16),
                nn.LeakyReLU(0.1),
            )
            classifier_in = 64 + 16
        else:
            self.feature_head = None
            classifier_in = 64
        self.classifier = nn.Sequential(
            nn.Linear(classifier_in, 64),
            nn.LeakyReLU(0.1),
            nn.Dropout(dropout),
            nn.Linear(64, num_classes),
        )

    def forward(self, x, x_feat):
        x = self.features(x)
        # Attention exists in the checkpoint, but training forward did not apply it.
        x = self.pool(x)
        x = self.profile_head(x)
        if self.use_scalar_features:
            x_feat = self.feature_head(x_feat)
            x = torch.cat([x, x_feat], dim=1)
        return self.classifier(x)


with META_PATH.open("r", encoding="utf-8") as f:
    model_meta = json.load(f)

checkpoint = torch.load(MODEL_PATH, map_location=DEVICE)

model_channels = checkpoint.get("channels", model_meta.get("channels", ["z", "gap", "pad"]))
model_feature_cols = checkpoint.get("feature_cols", model_meta.get("feature_cols", FEATURE_COLS))
model_n_bins = int(checkpoint.get("n_bins", model_meta.get("n_bins", N_BINS)))
index_to_label = {int(k): v for k, v in model_meta["index_to_label"].items()}
num_classes = len(index_to_label)

if model_n_bins != N_BINS:
    raise ValueError(f"Model expects {model_n_bins} bins, but helper is configured for {N_BINS}.")
missing_model_feature_cols = [c for c in model_feature_cols if c not in FEATURE_COLS]
if missing_model_feature_cols:
    raise ValueError(f"Model feature columns missing from notebook FEATURE_COLS: {missing_model_feature_cols}")
if list(model_channels) != ["z", "gap", "pad"]:
    raise ValueError(f"Unexpected model channels: {model_channels}")

z_mean = float(model_meta["z_mean"])
z_std = float(model_meta["z_std"])
feat_mean = np.asarray(model_meta["feat_mean"], dtype=np.float32)
feat_std = np.asarray(model_meta["feat_std"], dtype=np.float32)
feat_std[feat_std < 1e-8] = 1.0

model = RampCNN(
    in_channels=len(model_channels),
    num_features=len(model_feature_cols),
    num_classes=num_classes,
).to(DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print(f"Loaded model: {MODEL_PATH}")
print(f"Loaded metadata: {META_PATH}")
print(f"Device: {DEVICE}")
print(f"Channels: {model_channels}")
print(f"Feature columns: {model_feature_cols}")
print(f"Bins: {model_n_bins}")
print(f"Classes: {index_to_label}")
print(f"z_mean={z_mean:.6f}, z_std={z_std:.6f}")
print(f"feat_mean={feat_mean}")
print(f"feat_std={feat_std}")
print(f"Best epoch: {model_meta.get('best_epoch')} | validation_macro_f1={model_meta.get('validation_macro_f1')}")


In [ ]:
# ============================================================
# CELL 6 - Run CNN Predictions and Recombine Passthrough Rows
# ============================================================
def prediction_frame_to_tensors(frame):
    z_arr = frame[PROFILE_COLS].to_numpy(dtype=np.float32)
    gap_arr = frame[GAP_COLS].to_numpy(dtype=np.float32)
    pad_arr = frame[PAD_COLS].to_numpy(dtype=np.float32)
    x_profile = np.stack([z_arr, gap_arr, pad_arr], axis=1)
    x_feat = frame[model_feature_cols].to_numpy(dtype=np.float32) if model_feature_cols else np.zeros((len(frame), 0), dtype=np.float32)

    x_profile[:, 0, :] = (x_profile[:, 0, :] - z_mean) / z_std
    x_feat = (x_feat - feat_mean) / feat_std
    return (
        torch.tensor(x_profile, dtype=torch.float32, device=DEVICE),
        torch.tensor(x_feat, dtype=torch.float32, device=DEVICE),
    )

if df_model_input.empty:
    df_model_predictions = df_model_input.copy()
    for col in ["pred_label", "pred_class", "pred_confidence", "prob_RAMP", "prob_CURB_NO_RAMP", "prob_DEPRESSED_DITCH", "prediction_source"]:
        df_model_predictions[col] = pd.Series(dtype=object)
else:
    x_profile_t, x_feat_t = prediction_frame_to_tensors(df_model_input)
    with torch.no_grad():
        logits = model(x_profile_t, x_feat_t)
        probs = torch.softmax(logits, dim=1).detach().cpu().numpy()

    pred_idx = probs.argmax(axis=1).astype(int)
    pred_labels = [index_to_label[int(i)] for i in pred_idx]
    pred_classes = [CLASS_CODE[label] for label in pred_labels]

    df_model_predictions = df_model_input.copy()
    df_model_predictions["pred_label"] = pred_labels
    df_model_predictions["pred_class"] = pred_classes
    df_model_predictions["pred_confidence"] = probs.max(axis=1)
    for idx, label in index_to_label.items():
        df_model_predictions[f"prob_{label}"] = probs[:, int(idx)]
    df_model_predictions["prediction_source"] = "cnn_model"

prediction_cols = [
    "pred_label",
    "pred_class",
    "pred_confidence",
    "prob_RAMP",
    "prob_CURB_NO_RAMP",
    "prob_DEPRESSED_DITCH",
    "prediction_source",
]
for col in prediction_cols:
    if col not in df_model_predictions.columns:
        df_model_predictions[col] = np.nan
    if col not in df_passthrough.columns:
        df_passthrough[col] = np.nan

df_station_predictions = pd.concat([df_model_predictions, df_passthrough], ignore_index=True)
df_station_predictions = df_station_predictions.sort_values(["s_key", "side"]).reset_index(drop=True)
df_station_predictions["pred_class"] = df_station_predictions["pred_class"].astype(int)

print(f"CNN prediction rows      : {len(df_model_predictions):,}")
print(f"Passthrough rows         : {len(df_passthrough):,}")
print(f"Final station predictions: {len(df_station_predictions):,}")
print()
print("Prediction source counts:")
print(df_station_predictions["prediction_source"].value_counts(dropna=False).to_string())
print()
print("Predicted label distribution:")
print(df_station_predictions["pred_label"].value_counts(dropna=False).to_string())
print()
print("Original vs predicted station-side crosstab:")
print(pd.crosstab(df_station_predictions["original_label"], df_station_predictions["pred_label"], dropna=False).to_string())
print()
print("Confidence summary by predicted label:")
print(df_station_predictions.groupby("pred_label")["pred_confidence"].describe().round(4).to_string())

df_station_predictions[["s_m", "side", "original_label", "pred_label", "pred_class", "pred_confidence", "prediction_source"]].head(10)


In [ ]:
# ============================================================
# CELL 7 - Save Station-Side Prediction CSV
# ============================================================
station_prediction_cols = [
    "s_m",
    "s_key",
    "side",
    "source_file",
    "original_class",
    "original_label",
    "feature_rule_class",
    "feature_rule_label",
    "pred_class",
    "pred_label",
    "pred_confidence",
    "prob_RAMP",
    "prob_CURB_NO_RAMP",
    "prob_DEPRESSED_DITCH",
    "prediction_source",
    "feat_kink_dz",
    "feat_kink_slope",
    "profile_mode",
    "sidewalk_edge_buffer_m",
    "buffer_applied",
    "buffer_start_v",
    "buffer_end_v",
    "buffer_width_m",
    "sidewalk_edge_center_v",
    "sidewalk_edge_center_bin",
    "lane_edge_v",
    "side_edge_v",
    "interface_center_v",
    "n_valid_bins",
    "last_valid_bin",
    "n_sidewalk_pts",
    "n_lane_pts",
]
missing_station_cols = [c for c in station_prediction_cols if c not in df_station_predictions.columns]
if missing_station_cols:
    raise ValueError(f"Missing station prediction columns: {missing_station_cols}")

df_station_predictions_out = df_station_predictions[station_prediction_cols].copy()
df_station_predictions_out = df_station_predictions_out.sort_values(["s_key", "side"]).reset_index(drop=True)
df_station_predictions_out.to_csv(STATION_PREDICTIONS_CSV, index=False)

print(f"Station-side prediction CSV saved -> {STATION_PREDICTIONS_CSV}")
print(f"Rows written: {len(df_station_predictions_out):,}")
print()
print("Saved prediction source counts:")
print(df_station_predictions_out["prediction_source"].value_counts(dropna=False).to_string())
print()
print("Saved predicted label counts:")
print(df_station_predictions_out["pred_label"].value_counts(dropna=False).to_string())

df_station_predictions_out.head()


In [ ]:
# ============================================================
# CELL 8 - Apply Station-Side Predictions to Sidewalk Points
# ============================================================
# Original Stage 1 columns remain untouched. Prediction columns are added
# separately, and pred_R/G/B are the colors used for prediction outputs.
df_sw_pred = apply_predictions_to_sidewalk(df_sw, df_station_predictions_out)

required_point_pred_cols = [
    "pred_class",
    "pred_label",
    "pred_confidence",
    "prob_RAMP",
    "prob_CURB_NO_RAMP",
    "prob_DEPRESSED_DITCH",
    "pred_R",
    "pred_G",
    "pred_B",
    "pred_color",
]
missing_point_pred_cols = [c for c in required_point_pred_cols if c not in df_sw_pred.columns]
if missing_point_pred_cols:
    raise ValueError(f"Missing sidewalk prediction columns: {missing_point_pred_cols}")

unmatched_points = df_sw_pred["pred_label"].isna().sum()
if unmatched_points:
    raise ValueError(f"{unmatched_points:,} sidewalk points did not receive a station-side prediction.")

df_sw_pred["pred_class"] = df_sw_pred["pred_class"].astype(int)
df_sw_pred["pred_R"] = df_sw_pred["pred_R"].astype(int)
df_sw_pred["pred_G"] = df_sw_pred["pred_G"].astype(int)
df_sw_pred["pred_B"] = df_sw_pred["pred_B"].astype(int)

print(f"Sidewalk rows with predictions: {len(df_sw_pred):,}")
print(f"Unmatched sidewalk rows       : {unmatched_points:,}")
print()
print("Original point label distribution:")
print(df_sw_pred["label"].value_counts(dropna=False).to_string())
print()
print("Predicted point label distribution:")
print(df_sw_pred["pred_label"].value_counts(dropna=False).to_string())
print()
print("Original vs predicted point-label crosstab:")
print(pd.crosstab(df_sw_pred["label"], df_sw_pred["pred_label"], dropna=False).to_string())

preview_cols = [
    "x",
    "y",
    "z",
    "s_m",
    "v_m",
    "side",
    "class",
    "label",
    "R",
    "G",
    "B",
    "pred_class",
    "pred_label",
    "pred_confidence",
    "pred_R",
    "pred_G",
    "pred_B",
]
df_sw_pred[preview_cols].head()


In [ ]:
# ============================================================
# CELL 9 - Save Sidewalk Prediction CSV, ASC, and LAS
# ============================================================
# CSV keeps original Stage 1 class/label/color columns and adds pred_* columns.
# ASC/LAS use pred_R/pred_G/pred_B as R/G/B so CloudCompare shows CNN predictions.
csv_path = write_sidewalk_prediction_csv(df_sw_pred, SIDEWALK_PRED_CSV)
asc_path = write_sidewalk_prediction_asc(df_sw_pred, SIDEWALK_PRED_ASC)
las_path = write_sidewalk_prediction_las(df_sw_pred, SIDEWALK_PRED_LAS)

print(f"Sidewalk prediction CSV saved -> {csv_path}")
print(f"Sidewalk prediction ASC saved -> {asc_path}")
print(f"Sidewalk prediction LAS saved -> {las_path}")
print(f"Rows written: {len(df_sw_pred):,}")
print()
print("ASC/LAS color columns use model prediction RGB:")
print("  R <- pred_R")
print("  G <- pred_G")
print("  B <- pred_B")
print()
print("Files in predictions folder so far:")
for path in sorted(PREDICTIONS_DIR.glob("*")):
    print(f"  {path.name}")


In [ ]:
# ============================================================
# CELL 10 - Single Station-Side Debug Viewer
# ============================================================
# Change these two values and rerun this cell to inspect any station-side.
DEBUG_S_M = 9.8
DEBUG_SIDE = "RIGHT"  # RIGHT or LEFT

debug_side = str(DEBUG_SIDE).strip().upper()
if debug_side not in {"RIGHT", "LEFT"}:
    raise ValueError("DEBUG_SIDE must be RIGHT or LEFT")

same_side = df_station_predictions_out[df_station_predictions_out["side"] == debug_side].copy()
if same_side.empty:
    raise ValueError(f"No station predictions found for side={debug_side}")
nearest_idx = (same_side["s_m"].astype(float) - float(DEBUG_S_M)).abs().idxmin()
pred_row = df_station_predictions_out.loc[nearest_idx]
s_key_debug = round(float(pred_row["s_key"]), 3)

input_match = df_pred_input[(df_pred_input["s_key"] == s_key_debug) & (df_pred_input["side"] == debug_side)]
if input_match.empty:
    raise ValueError(f"No model-input row found for s_key={s_key_debug}, side={debug_side}")
input_row = input_match.iloc[0]

sw_pts = df_sw[(df_sw["_s_key"] == s_key_debug) & (df_sw["side"] == debug_side)].copy()
lane_pts = df_lane[(df_lane["_s_key"] == s_key_debug) & (df_lane["side"] == debug_side)].copy()
point_preds = df_sw_pred[(pd.to_numeric(df_sw_pred["s_m"], errors="coerce").round(3) == s_key_debug) & (df_sw_pred["side"] == debug_side)]

z_profile = input_row[PROFILE_COLS].to_numpy(dtype=float)
gap_profile = input_row[GAP_COLS].to_numpy(dtype=float)
pad_profile = input_row[PAD_COLS].to_numpy(dtype=float)
v_axis = np.arange(N_BINS) * 0.08
last_valid_bin = int(input_row["last_valid_bin"])
valid_slice = slice(0, last_valid_bin + 1) if last_valid_bin >= 0 else slice(0, 0)

if sw_pts.empty:
    sidewalk_start_v = np.nan
    debug_x0 = 0.0
    debug_x1 = min(float(v_axis[-1]), 2.0)
else:
    sidewalk_start_v = float(np.nanmin(np.abs(sw_pts["v_m"].to_numpy(dtype=float))))
    debug_x0 = max(0.0, sidewalk_start_v - 1.0)
    debug_x1 = min(float(v_axis[-1]), sidewalk_start_v + 1.0)

print(f"Inspecting s = {float(pred_row['s_m']):.3f} m | side = {debug_side} | requested S_DEBUG = {DEBUG_S_M}")
print()
print("Prediction summary:")
print(f"  original_label      : {pred_row['original_label']} ({pred_row['original_class']})")
print(f"  feature_rule_label  : {pred_row['feature_rule_label']} ({pred_row['feature_rule_class']})")
print(f"  pred_label          : {pred_row['pred_label']} ({pred_row['pred_class']})")
print(f"  pred_confidence     : {float(pred_row['pred_confidence']):.4f}")
print(f"  prediction_source   : {pred_row['prediction_source']}")
print(f"  prob_RAMP           : {float(pred_row['prob_RAMP']):.4f}")
print(f"  prob_CURB_NO_RAMP   : {float(pred_row['prob_CURB_NO_RAMP']):.4f}")
print(f"  prob_DEPRESSED_DITCH: {float(pred_row['prob_DEPRESSED_DITCH']):.4f}")
print()
print("Feature/profile summary:")
print(f"  feat_kink_dz        : {pred_row['feat_kink_dz']}")
print(f"  feat_kink_slope     : {pred_row['feat_kink_slope']}")
print(f"  n_valid_bins        : {int(input_row['n_valid_bins'])}")
print(f"  last_valid_bin      : {last_valid_bin}")
print(f"  n_sidewalk_pts      : {len(sw_pts):,}")
print(f"  n_lane_pts          : {len(lane_pts):,}")
print(f"  sidewalk_start_v    : {sidewalk_start_v:.3f} m")
print(f"  debug x-window      : [{debug_x0:.3f}, {debug_x1:.3f}] m")
print(f"  sidewalk pred labels: {point_preds['pred_label'].value_counts(dropna=False).to_dict()}")

fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=False)

ax = axes[0]
if not lane_pts.empty:
    ax.scatter(np.abs(lane_pts["v_m"].to_numpy(dtype=float)), lane_pts["z"].to_numpy(dtype=float), s=3, alpha=0.25, label="lane raw")
if not sw_pts.empty:
    ax.scatter(np.abs(sw_pts["v_m"].to_numpy(dtype=float)), sw_pts["z"].to_numpy(dtype=float), s=3, alpha=0.25, label="sidewalk raw")
if last_valid_bin >= 0:
    ax.plot(v_axis[valid_slice], z_profile[valid_slice], color="black", lw=2.0, label="model z profile")
    ax.axvline(v_axis[last_valid_bin], color="red", lw=1.0, ls="--", label="last valid bin")
if np.isfinite(sidewalk_start_v):
    ax.axvline(sidewalk_start_v, color="purple", lw=1.5, ls=":", label="sidewalk start")
ax.set_xlim(debug_x0, debug_x1)
z_window_values = []
if not lane_pts.empty:
    lane_v_abs = np.abs(lane_pts["v_m"].to_numpy(dtype=float))
    lane_z_vals = lane_pts["z"].to_numpy(dtype=float)
    z_window_values.extend(lane_z_vals[(lane_v_abs >= debug_x0) & (lane_v_abs <= debug_x1) & np.isfinite(lane_z_vals)].tolist())
if not sw_pts.empty:
    sw_v_abs = np.abs(sw_pts["v_m"].to_numpy(dtype=float))
    sw_z_vals = sw_pts["z"].to_numpy(dtype=float)
    z_window_values.extend(sw_z_vals[(sw_v_abs >= debug_x0) & (sw_v_abs <= debug_x1) & np.isfinite(sw_z_vals)].tolist())
profile_mask = (v_axis >= debug_x0) & (v_axis <= debug_x1) & np.isfinite(z_profile)
z_window_values.extend(z_profile[profile_mask].tolist())
if z_window_values:
    z_min = float(np.nanmin(z_window_values))
    z_max = float(np.nanmax(z_window_values))
    z_pad = max(0.02, 0.10 * max(z_max - z_min, 1e-6))
    ax.set_ylim(z_min - z_pad, z_max + z_pad)
ax.set_title(f"Station-side profile | s={float(pred_row['s_m']):.3f} | {debug_side} | pred={pred_row['pred_label']} ({float(pred_row['pred_confidence']):.3f})")
ax.set_xlabel("abs(v_m) lateral distance (m)")
ax.set_ylabel("z (m)")
ax.grid(True, alpha=0.3)
ax.legend(loc="best", fontsize=9)

ax = axes[1]
ax.plot(v_axis, gap_profile, color="orange", lw=1.5, label="gap flag")
ax.plot(v_axis, pad_profile, color="gray", lw=1.5, label="pad flag")
if last_valid_bin >= 0:
    ax.axvspan(0.0, v_axis[last_valid_bin], color="green", alpha=0.08, label="valid extent")
    ax.axvline(v_axis[last_valid_bin], color="red", lw=1.0, ls="--")
if np.isfinite(sidewalk_start_v):
    ax.axvline(sidewalk_start_v, color="purple", lw=1.5, ls=":", label="sidewalk start")
ax.set_xlim(debug_x0, debug_x1)
ax.set_xlabel("abs(v_m) lateral distance (m)")
ax.set_ylabel("flag value")
ax.set_ylim(-0.05, 1.05)
ax.grid(True, alpha=0.3)
ax.legend(loc="best", fontsize=9)

plt.tight_layout()
plt.show()
